# Method experiment C1: duplicate-aware (group-balanced) training on the v2 clean folds

Hypothesis: in v2, 56 of 282 groups (8+ images each) hold 36% of all images, so training weights every image equally while the test counts every group once. Group-balanced training caps large groups (12 images) and repeats small groups (up to x4) so every group has a similar weight, with the number of optimisation steps matched to the baseline (epochs scaled by images_before/images_after).
Gate (pre-registered, docs/METHOD_DESIGN.md): paired by fold against the plain baseline with the same seeds, mean gain >= +0.9 pt mAP50-95 on v2 (3 seeds x 5 folds), >= 4 of 5 folds positive. 15 runs per part.
* **PART = "A"**: YOLOv8s + group balance, seeds 42/43/44 (about 3.5 h on T4 x2)
* **PART = "B"**: YOLO11s + group balance, seeds 42/43/44 (about 3.5 h on T4 x2)
Start with SMOKE = True (2 epochs, 1 run). GPU T4 x2, Internet ON.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215"], check=True)
os.environ["WANDB_MODE"] = "disabled"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
SMOKE = False
PART = "A"                     # "A" = YOLOv8s, "B" = YOLO11s
MODEL = {"A": "yolov8s.pt", "B": "yolo11s.pt"}[PART]
SEEDS = [42, 43, 44]
FOLDS = [0, 1, 2, 3, 4]
SPLITS = f"{REPO}/data_splits_v2"
assert os.path.isdir(f"{SPLITS}/clean5"), "data_splits_v2 missing: push it and re-run the clone cell"
if SMOKE:
    EPOCHS, OUT = 2, f"/kaggle/working/results/gb_{PART}_smoke"; FOLDS = [0]; SEEDS = [42]
else:
    EPOCHS, OUT = 100, f"/kaggle/working/results/gb_{PART}"

def make_cmd(model, fold, seed):
    def f(gpu):
        return [sys.executable, f"{REPO}/scripts/training/run_baseline.py", "--mode", "clean", "--fold", str(fold), "--seed", str(seed),
                "--model", model, "--epochs", str(EPOCHS), "--fixed-epochs", "--device", str(gpu), "--data-root", DATA_ROOT,
                "--splits-dir", SPLITS, "--out", OUT, "--arm", "gb", "--group-balance", "--gb-match-steps"]
    return f
TASKS = [(f"gbclean_f{f}_s{s}_{MODEL[:-3]}", make_cmd(MODEL, f, s), None) for s in SEEDS for f in FOLDS]
print(len(TASKS), "runs, epochs =", EPOCHS, "->", OUT)

In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count()))
q = queue.Queue()
for t in TASKS:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            name, cmd_fn, needs = q.get_nowait()
        except queue.Empty:
            return
        while needs and not os.path.exists(needs):          # e.g. arm 6 waits for the FracAtlas pre-trained checkpoint
            if any(n.startswith("fa_pre_") for n in failed):
                break
            time.sleep(20)
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd_fn(gpu), stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))

In [ ]:
import shutil, glob, json
rows = []
for mp in sorted(glob.glob(f"{OUT}/*/metrics.json")):
    m = json.load(open(mp)); rows.append((m["tag"], m["epochs_cfg"], round(m["test"]["map5095"], 4), round(m["test"]["macro_eligible_ap5095"], 4), (m.get("group_balance") or {}).get("images_after")))
print("tag | epochs | test mAP50-95 | macro | train images after balance")
for r in rows: print(*r, sep=" | ")
if rows: print("mean mAP50-95 over runs:", round(sum(r[2] for r in rows) / len(rows), 4))
shutil.make_archive(f"/kaggle/working/gb_{PART}_results", "zip", OUT)
print(sorted(os.listdir(OUT)))
if failed:
    print("\nWARNING: failed runs:", failed)

After the run: download `gb_<PART>_results.zip`, unzip once into `results/gb_<PART>/` (rename the unzipped folder), commit and push; then `python scripts/evaluation/compare_method.py --arm gb` (needs the v2 baseline seeds 42-44 from notebooks 10 and 11).